In [6]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from IPython.display import clear_output
import time
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import Counter
from Buy_Presure_Scaner import *
from Volatile_4h_Scaner import *
from Gainer_List import *
from Consolidation_24_Scaner import *
from RSI_30_Scaner import *
from Fetch_Live_Data import *
from Chandelier_ZLSMA import *

In [9]:
binance_df = get_binance_buy_presure(min_volume_usdt=1000000, top_n=50)
binance_df

,rank,symbol,current_buy_pressure,current_sell_pressure,avg_buy_pressure_5periods,buy_pressure_trend,latest_price_change,latest_volume,volume_trend,volatility,timestamp,signal,momentum
0,1,IMXUSDT,82.16,17.84,61.57,7.15,0.00,7.591380e+03,-24.01,1.10,2025-06-12T18:04:43.111017,AVOID,STRONG
1,2,VETUSDT,86.74,13.26,60.19,11.43,0.20,8.767243e+05,-14.97,0.97,2025-06-12T18:04:43.182659,AVOID,STRONG
2,3,SHIBUSDT,80.23,19.77,60.16,4.06,-0.08,3.653945e+09,-15.55,0.81,2025-06-12T18:04:42.158747,AVOID,GOOD
3,4,ENSUSDT,56.49,43.51,58.25,-0.97,0.18,1.262560e+03,-16.20,1.20,2025-06-12T18:04:42.485657,AVOID,BUILDING
4,5,AIXBTUSDT,59.73,40.27,56.79,3.38,0.00,9.845410e+04,7.78,1.69,2025-06-12T18:04:42.503262,BUY,EXPLOSIVE
5,6,KMNOUSDT,54.76,45.24,56.56,0.34,0.08,6.775500e+04,5.36,1.11,2025-06-12T18:04:43.059882,BUY,GOOD
6,7,BOMEUSDT,49.43,50.57,56.36,-1.12,0.00,1.502274e+07,11.34,1.10,2025-06-12T18:04:43.140662,AVOID,GOOD
7,8,JTOUSDT,75.56,24.44,56.16,2.83,0.00,1.321300e+03,61.92,1.41,2025-06-12T18:04:42.758419,BUY,EXPLOSIVE
8,9,AXLUSDT,76.02,23.98,56.03,5.91,-0.34,1.142553e+05,-13.16,1.76,2025-06-12T18:04:42.775269,AVOID,GOOD
9,10,ETHFIUSDT,58.62,41.38,55.76,-0.25,0.16,8.445080e+04,-15.07,1.54,2025-06-12T18:04:41.758163,AVOID,BUILDING


In [10]:
def get_max_buy_pressure_trend_symbols(binance_df_1_hr):
    """
    Filter cryptocurrency data and return symbols with maximum buy pressure trend.
    
    Parameters:
    binance_df_1_hr (DataFrame): Input DataFrame with cryptocurrency data
    
    Returns:
    numpy.ndarray: Array of symbols with maximum buy pressure trend
    """
    # Define stablecoins to filter out
    stablecoins = [ 
        'USDTUSDT', 'USDCUSDT', 'BUSDUSDT', 'TUSDUSDT', 'FDUSDUSDT', 'USDPUSDT', 'DAIUSDT', 'USD1USDT', 'PYUSDUSDT', 
        'GUSDUSDT', 'FRAXUSDT', 'USDDUSDT', 'MIMUSDT', 'LUSDUSDT', 'FEIUSDT', 'HUSDUSDT', 'SUSDUSDT', 'OUSDUSDT', 
        'USTCUSDT', 'VAIUSDT', 'DOLAUSDT', 'ALUSDUSDT', 'MUSDUSDT', 'DUSDUSDT', 'CUSDUSDT', 'NUSDUSDT', 'ZUSDUSDT' 
    ]
    
    # Apply filters
    binance_df_1_hr_filtered = binance_df_1_hr[ 
        (binance_df_1_hr['current_buy_pressure'] > binance_df_1_hr['current_sell_pressure']) &  
        (binance_df_1_hr['momentum'] != "BUILDING") &  
        (binance_df_1_hr['buy_pressure_trend'] > 1) &  
        (binance_df_1_hr['volume_trend'] > 1) 
    ]
    
    # Select relevant columns
    binance_df_1_hr_filtered = binance_df_1_hr_filtered[["symbol", "current_buy_pressure", "avg_buy_pressure_5periods", "buy_pressure_trend", "signal", "momentum"]]
    
    # Filter out stablecoins
    binance_df_1_hr_filtered = binance_df_1_hr_filtered[~binance_df_1_hr_filtered['symbol'].isin(stablecoins)]

    # Check if any data remains after filtering
    if binance_df_1_hr_filtered.empty:
        return []
    
    print(binance_df_1_hr_filtered)
    print("\n")
    
    # Find maximum buy pressure trend
    max_buy_pressure_trend = binance_df_1_hr_filtered['buy_pressure_trend'].max()
    max_buy_pressure_trend_row = binance_df_1_hr_filtered[binance_df_1_hr_filtered['buy_pressure_trend'] == max_buy_pressure_trend]
    print(max_buy_pressure_trend_row)
    print("\n")
    # Return symbol values
    return max_buy_pressure_trend_row['symbol'].values

In [12]:
symbols = get_max_buy_pressure_trend_symbols(binance_df)

       symbol  current_buy_pressure  avg_buy_pressure_5periods  \
4   AIXBTUSDT                 59.73                      56.79   
7     JTOUSDT                 75.56                      56.16   
12    SSVUSDT                 90.25                      55.02   
14    ETHUSDT                 62.55                      54.71   
25    FETUSDT                 67.43                      53.16   
32    APTUSDT                 58.12                      52.13   
36    INJUSDT                 56.73                      51.70   
43    SOLUSDT                 84.78                      50.85   
47    WIFUSDT                 60.77                      50.56   

    buy_pressure_trend signal   momentum  
4                 3.38    BUY  EXPLOSIVE  
7                 2.83    BUY  EXPLOSIVE  
12               10.28    BUY  EXPLOSIVE  
14                2.97  WATCH  EXPLOSIVE  
25                5.12  WATCH  EXPLOSIVE  
32                2.22  WATCH     STRONG  
36                5.10  WATCH  EXPLOSI

In [13]:
symbols

array(['SOLUSDT'], dtype=object)

In [ ]:
# Define your function to fetch, calculate, and merge the data
def fetch_and_process_data(symbol):
    df = get_single_fetch(symbol, 100)  # Fetch data
    df_zlsma = calculate_zlsma(df, close_column='close', length=200)  # Calculate ZLSMA
    df_chandelier = calculate_chandelier(df, atr_period=1, atr_multiplier=2.0)  # Calculate Chandelier
    merged_chandelier_zlsma = merge_zlsma_chandelier(df_zlsma, df_chandelier)  # Merge ZLSMA and Chandelier
    merged_chandelier_zlsma = merged_chandelier_zlsma[["timestamp", "close", "zlsma_200", "buy_signal", "sell_signal"]]  # Filter relevant columns
    return merged_chandelier_zlsma[90:]

# Run the function every 10 seconds
while True:
    clear_output(wait=True)  # Clear previous output in Jupyter Notebook
    print("Monitoring on: ", symbols)
    result = fetch_and_process_data(symbols)
    
    # Now only print the relevant output
    print(result)  # Display the new result
    time.sleep(10)  # Wait for 10 seconds before running the next iteration

Monitoring on:  ['SOLUSDT']
Valid zlsma_200 values: 36 out of 100 rows.
             timestamp   close   zlsma_200  buy_signal  sell_signal
90 2025-06-12 15:45:00  159.32  159.397831           0            1
91 2025-06-12 16:00:00  159.49  159.249928           0            1
92 2025-06-12 16:15:00  158.87  159.010671           0            1
93 2025-06-12 16:30:00  158.81  158.859766           0            1
94 2025-06-12 16:45:00  158.80  158.738526           0            1
95 2025-06-12 17:00:00  159.34  158.780249           0            1
96 2025-06-12 17:15:00  159.05  158.738327           0            1
97 2025-06-12 17:30:00  159.19  158.704807           0            1
98 2025-06-12 17:45:00  159.46  158.797034           0            1
99 2025-06-12 18:00:00  159.57  158.928734           0            1


In [8]:
oversold_15m = get_low_rsi_coins(rsi_threshold=30, interval="15m", lookback_hours=4)
oversold_15m

🚀 Fetching coins with RSI below 30
📊 Interval: 15m, Lookback: 4 hours
📈 Analyzing 403 USDT pairs...
✓ WINUSDT: RSI 29.46, 4h Change: -1.13%
   Processed 100/403 coins... (Errors: 0)
   Processed 200/403 coins... (Errors: 0)
✓ SYNUSDT: RSI 28.36, 4h Change: -3.47%
   Processed 300/403 coins... (Errors: 0)
✓ POLUSDT: RSI 29.68, 4h Change: -2.63%
   Processed 400/403 coins... (Errors: 0)

✅ Found 3 coins with RSI below 30
📊 Analyzed 403 coins with 0 errors


,symbol,coin,current_price,rsi,interval,lookback_hours,change_4h,volume_4h,high_4h,low_4h,price_from_low,price_from_high
0,SYNUSDT,SYN,0.153000,28.361139,15m,4,-3.470032,1.076003e+06,0.158600,0.152600,0.262123,-3.530895
1,WINUSDT,WIN,0.000052,29.455361,15m,4,-1.126814,1.473403e+09,0.000053,0.000052,0.271160,-1.521780
2,POLUSDT,POL,0.221900,29.678302,15m,4,-2.632734,8.344538e+06,0.228400,0.221100,0.361827,-2.845884


In [ ]:
df_zlsma = calculate_zlsma_from_csv('df_2025_05_1_to_22.csv', close_column='close', length=200)
df_chandelier = calculate_chandelier_from_csv('df_2025_05_1_to_22.csv', atr_period=1, atr_multiplier=2.0)
merged_chandelier_zlsma = merge_chandelier_with_zlsma(df_chandelier, df_zlsma)

In [10]:
import requests
import time
import json
from datetime import datetime

def fetch_binance_15min_candles(symbol="BTCUSDT", limit=15):
    """
    Fetch 15-minute OHLCV candlestick data from Binance API
    
    Args:
        symbol (str): Trading pair symbol (default: BTCUSDT)
        limit (int): Number of candles to fetch (default: 15)
    
    Returns:
        list: List of OHLCV candlestick data or None if error
    """
    base_url = "https://api.binance.com/api/v3/klines"
    
    params = {
        'symbol': symbol,
        'interval': '15m',
        'limit': limit
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=10)
        response.raise_for_status()
        
        data = response.json()
        
        # Format the data as OHLCV
        ohlcv_data = []
        for candle in data:
            ohlcv_candle = {
                'timestamp': datetime.fromtimestamp(candle[0] / 1000).strftime('%Y-%m-%d %H:%M:%S'),
                'open': float(candle[1]),
                'high': float(candle[2]),
                'low': float(candle[3]),
                'close': float(candle[4]),
                'volume': float(candle[5])
            }
            ohlcv_data.append(ohlcv_candle)
        
        return ohlcv_data
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return None

def live_candle_monitor(symbol="BTCUSDT", limit=15, interval_seconds=10):
    """
    Continuously fetch and display live 15-minute OHLCV candle data every 10 seconds
    
    Args:
        symbol (str): Trading pair symbol
        limit (int): Number of candles to fetch
        interval_seconds (int): Fetch interval in seconds
    """
    print(f"Starting live OHLCV monitoring for {symbol} - 15min candles")
    print(f"Fetching {limit} candles every {interval_seconds} seconds")
    print("Press Ctrl+C to stop\n")
    
    try:
        while True:
            print(f"\n--- Fetching data at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ---")
            
            candles = fetch_binance_15min_candles(symbol, limit)
            
            if candles:
                # Display the latest candle
                latest_candle = candles[-1]
                print(f"Latest 15min OHLCV candle for {symbol}:")
                print(f"Timestamp: {latest_candle['timestamp']}")
                print(f"Open (O): ${latest_candle['open']:.2f}")
                print(f"High (H): ${latest_candle['high']:.2f}")
                print(f"Low (L): ${latest_candle['low']:.2f}")
                print(f"Close (C): ${latest_candle['close']:.2f}")
                print(f"Volume (V): {latest_candle['volume']:.2f}")
                
                # Optionally, you can process all candles here
                # for candle in candles:
                #     print(candle)
            else:
                print("Failed to fetch candle data")
            
            time.sleep(interval_seconds)
            
    except KeyboardInterrupt:
        print("\nStopping live monitor...")

def get_single_fetch(symbol="BTCUSDT", limit=15):
    """
    Get a single fetch of 15-minute OHLCV candle data
    
    Args:
        symbol (str): Trading pair symbol
        limit (int): Number of candles to fetch
    
    Returns:
        list: OHLCV candlestick data
    """
    return fetch_binance_15min_candles(symbol, limit)

# Example usage
if __name__ == "__main__":
    # Option 1: Single OHLCV fetch
    print("Single OHLCV fetch example:")
    data = get_single_fetch("BTCUSDT", 15)
    if data:
        print(f"Fetched {len(data)} OHLCV candles")
        print("Latest OHLCV candle:", data[-1])
    
    print("\n" + "="*50 + "\n")
    
    # Option 2: Continuous monitoring
    # Uncomment the line below to start live monitoring
    # live_candle_monitor("BTCUSDT", 15, 10)

Single OHLCV fetch example:
Fetched 15 OHLCV candles
Latest OHLCV candle: {'timestamp': '2025-06-12 15:45:00', 'open': 107966.79, 'high': 107966.79, 'low': 107896.6, 'close': 107912.18, 'volume': 10.06709}




In [11]:
live_candle_monitor("BTCUSDT", 15, 10)

Starting live OHLCV monitoring for BTCUSDT - 15min candles
Fetching 15 candles every 10 seconds
Press Ctrl+C to stop


--- Fetching data at 2025-06-12 15:49:13 ---
Latest 15min OHLCV candle for BTCUSDT:
Timestamp: 2025-06-12 15:45:00
Open (O): $107966.79
High (H): $107966.79
Low (L): $107896.60
Close (C): $107900.01
Volume (V): 11.89

--- Fetching data at 2025-06-12 15:49:23 ---
Latest 15min OHLCV candle for BTCUSDT:
Timestamp: 2025-06-12 15:45:00
Open (O): $107966.79
High (H): $107966.79
Low (L): $107896.60
Close (C): $107900.00
Volume (V): 12.39

--- Fetching data at 2025-06-12 15:49:34 ---
Latest 15min OHLCV candle for BTCUSDT:
Timestamp: 2025-06-12 15:45:00
Open (O): $107966.79
High (H): $107966.79
Low (L): $107896.60
Close (C): $107900.01
Volume (V): 12.76

--- Fetching data at 2025-06-12 15:49:44 ---

Stopping live monitor...


In [ ]:
analyzer = ConsolidationAnalyzer()
non_consolidating_coins = analyzer.analyze_all_coins( 
        top_count=250  # Get top 100, display top 25
    )
con_df = analyzer.display_results(non_consolidating_coins, show_count=100)
con_df

Step 1: Fetching all trading pairs...
Found 403 active USDT trading pairs
Step 2: Fetching 24h ticker data...
Step 3: Analyzing 403 coins for consolidation...
Progress: 50/403 coins analyzed...
Progress: 100/403 coins analyzed...
Progress: 150/403 coins analyzed...
Progress: 200/403 coins analyzed...
Progress: 250/403 coins analyzed...
Progress: 300/403 coins analyzed...
Progress: 350/403 coins analyzed...
Progress: 400/403 coins analyzed...
Analysis complete! Processed 403 coins, 0 failed
Found 176 non-consolidating coins


,rank,symbol,price,24h_Change,24h_Range,volatility,ATR %,24h_Volume,vol_Surge
0,1,PEPEUSDT,$0.000012,-3.60%,10.16%,1.29%,1.93%,"$309,397,520",1.0x
1,2,KAIAUSDT,$0.167700,13.31%,19.56%,2.27%,3.56%,"$51,605,305",0.9x
2,3,HMSTRUSDT,$0.001108,-25.24%,39.17%,2.43%,4.01%,"$25,899,842",1.4x
3,4,TRXUSDT,$0.278000,-3.94%,5.72%,0.62%,0.72%,"$154,236,201",0.9x
4,5,RESOLVUSDT,$0.331300,9.92%,38.79%,2.89%,9.15%,"$54,111,655",0.4x
5,6,WIFUSDT,$0.942000,-6.92%,12.53%,1.44%,2.40%,"$76,014,709",1.0x
6,7,PENDLEUSDT,$3.908000,-11.92%,16.53%,1.26%,1.94%,"$27,919,868",1.4x
7,8,RVNUSDT,$0.021490,8.10%,22.43%,2.95%,4.55%,"$40,968,036",0.7x
8,9,ENAUSDT,$0.343900,-3.80%,10.73%,1.29%,2.15%,"$81,064,094",0.9x
9,10,ETHFIUSDT,$1.251000,-8.62%,12.71%,1.26%,2.31%,"$35,502,253",1.0x


In [7]:
calculator = BinanceVolatilityCalculator()
top_volatile = calculator.get_top_volatile_coins_4h(top_n=50)
top_volatile

Fetching last 4 hours of 15-minute data for volatility calculation...
Analysis time: 2025-06-12 11:20:27
Found 403 USDT pairs
Processed 20/403 pairs... (5.0%)
Processed 40/403 pairs... (9.9%)
Processed 60/403 pairs... (14.9%)
Processed 80/403 pairs... (19.9%)
Processed 100/403 pairs... (24.8%)
Processed 120/403 pairs... (29.8%)
Processed 140/403 pairs... (34.7%)
Processed 160/403 pairs... (39.7%)
Processed 180/403 pairs... (44.7%)
Processed 200/403 pairs... (49.6%)
Processed 220/403 pairs... (54.6%)
Processed 240/403 pairs... (59.6%)
Processed 260/403 pairs... (64.5%)
Processed 280/403 pairs... (69.5%)
Processed 300/403 pairs... (74.4%)
Processed 320/403 pairs... (79.4%)
Processed 340/403 pairs... (84.4%)
Processed 360/403 pairs... (89.3%)
Processed 380/403 pairs... (94.3%)
Processed 400/403 pairs... (99.3%)

Analysis complete! Found 400 qualifying coins.


,symbol,coin,current_price,start_price_4h,price_change_4h,volatility_score,std_volatility,price_range_volatility,max_single_move,max_drawdown_4h,volume_4h,volume_volatility,data_points,rank
402,RESOLVUSDT,RESOLV,3.438000e-01,3.851000e-01,10.724487,3.165264,5.361615,3.774451,6.030429,16.277298,1.471123e+08,0.012627,48,1
69,ARDRUSDT,ARDR,9.550000e-02,8.566000e-02,11.487275,3.046962,5.281250,1.288103,9.230406,3.114538,3.363698e+07,0.030856,48,2
47,RVNUSDT,RVN,2.130000e-02,2.210000e-02,3.619910,2.789526,5.408317,2.236924,6.422018,11.824755,9.779415e+08,0.016300,48,3
333,HMSTRUSDT,HMSTR,1.065000e-03,1.283000e-03,16.991426,2.408583,4.416645,1.926811,5.747126,18.139892,1.261090e+10,0.015592,48,4
361,ANIMEUSDT,ANIME,3.143000e-02,3.013000e-02,4.314637,1.858972,4.024325,1.880184,3.371522,4.623179,5.320202e+08,0.012996,48,5
56,FTTUSDT,FTT,9.703000e-01,9.592000e-01,1.157214,1.668508,3.157242,1.104118,4.251889,4.251889,1.826376e+06,0.015228,48,6
385,VIRTUALUSDT,VIRTUAL,2.071400e+00,2.248400e+00,7.872265,1.573198,2.938396,1.306222,3.640929,10.383317,1.566053e+07,0.008778,48,7
156,MASKUSDT,MASK,1.578000e+00,1.565000e+00,0.830671,1.536301,3.067381,1.438507,3.158560,7.880911,1.391644e+07,0.009065,48,8
107,FLMUSDT,FLM,3.690000e-02,3.750000e-02,1.600000,1.490171,2.659364,1.043112,3.835616,4.800000,4.024932e+07,0.008700,48,9
159,ATAUSDT,ATA,5.040000e-02,5.080000e-02,0.787402,1.464923,2.416251,0.903150,4.106776,5.458090,1.514126e+07,0.027749,48,10


In [14]:
def get_most_common_symbol_every_4_hours():
    while True:
        print("Fetching Buying Pressure")
        print("\n")
        # Fetch the data
        binance_df = get_binance_buy_presure(min_volume_usdt=1000000, top_n=50)
        binance_df_filtered = binance_df[(binance_df['current_buy_pressure'] > 
                                          binance_df['current_sell_pressure']) & 
                                         (binance_df['momentum'] != "BUILDING") & 
                                         (binance_df['buy_pressure_trend'] > 1) & 
                                         (binance_df['momentum'] != "BUILDING") & 
                                         (binance_df['buy_pressure_trend'] > 1) & 
                                         (binance_df['volume_trend'] > 1)] 
        print("Completed Buying Pressure Scan")
        print("\n")
        
        print("Fetching Volatile Coins")
        print("\n")
        calculator = BinanceVolatilityCalculator()
        top_volatile = calculator.get_top_volatile_coins_4h(top_n=50)
        print("Completed Volatile Coins Scan")
        print("\n")
        
        print("Fetching Top Gainers")
        print("\n")
        top_gainers = fetch_binance_gainers(limit=50)
        top_gainers = display_gainers(top_gainers)
        print("Completed Top Gainers Scan")
        print("\n")
        
        print("Fetching Consolidation Coins")
        print("\n")
        analyzer = ConsolidationAnalyzer()
        non_consolidating_coins = analyzer.analyze_all_coins( 
                top_count=250  # Get top 100, display top 25
            )
        con_df = analyzer.display_results(non_consolidating_coins, show_count=50)
        print("Completed Consolidation Coins Scan")

        # Convert each list of symbols to sets
        binance_symbols = set(binance_df_filtered['symbol'])
        volatile_symbols = set(top_volatile['symbol'])
        gainers_symbols = set(top_gainers['symbol'])
        con_df_symbols = set(con_df['symbol'])

        # Find the common symbols across all four lists
        common_symbols_in_4 = binance_symbols & volatile_symbols & gainers_symbols & con_df_symbols
        common_symbols_in_3 = binance_symbols & volatile_symbols & con_df_symbols

        # Combine all the symbols into a single list
        all_symbols = list(binance_symbols + volatile_symbols + gainers_symbols + con_df_symbols)
        
        # Count the occurrences of each symbol
        symbol_counter = Counter(all_symbols)
        
        # Find the most common symbol
        most_common_symbol = symbol_counter.most_common(1)[0]

        # Sleep for 4 hours (14400 seconds)
        #time.sleep(14400)  # 4 hours in seconds
        
        return most_common_symbol, common_symbols_in_4, common_symbols_in_3

In [15]:
most_common, common_in_4, common_in_3 = get_common_symbols_every_4_hours()

Fetching Buying Presure


Completed Buying Pressure Scan


Fetching Volatile Coins


Fetching last 4 hours of 15-minute data for volatility calculation...
Analysis time: 2025-06-11 19:13:30
Found 402 USDT pairs
Processed 20/402 pairs... (5.0%)
Processed 40/402 pairs... (10.0%)
Processed 60/402 pairs... (14.9%)
Processed 80/402 pairs... (19.9%)
Processed 100/402 pairs... (24.9%)
Processed 120/402 pairs... (29.9%)
Processed 140/402 pairs... (34.8%)
Processed 160/402 pairs... (39.8%)
Processed 180/402 pairs... (44.8%)
Processed 200/402 pairs... (49.8%)
Processed 220/402 pairs... (54.7%)
Processed 240/402 pairs... (59.7%)
Processed 260/402 pairs... (64.7%)
Processed 280/402 pairs... (69.7%)
Processed 300/402 pairs... (74.6%)
Processed 320/402 pairs... (79.6%)
Processed 340/402 pairs... (84.6%)
Processed 360/402 pairs... (89.6%)
Processed 380/402 pairs... (94.5%)
Processed 400/402 pairs... (99.5%)

Analysis complete! Found 399 qualifying coins.
Completed Volatile Coins Scan


Fetching Top G

ValueError: not enough values to unpack (expected 3, got 2)

In [11]:
common_in_4

set()

In [12]:
common_in_3

set()

🚀 Fetching coins with RSI below 25
📊 Interval: 15m, Lookback: 4 hours
📈 Analyzing 403 USDT pairs...
✓ ICXUSDT: RSI 22.78, 4h Change: -6.37%
✓ ENJUSDT: RSI 22.00, 4h Change: -4.06%


KeyboardInterrupt: 